# Treinamento ResNet50 Avançado (FER2013)

Este notebook foi estruturado para resolver a deficiência de aprendizado observada nas versões anteriores da ResNet50. Técnicas implementadas:
- Redimensionamento de 48x48 para 224x224 (para extração otimizada de features do ImageNet).
- Augmentation agressivo (rotação, shift, zoom, flip) focado em simular oclusões e variações de pose.
- Pesos de Classes (*Class Weights*) dinâmicos para contornar o desbalanceamento das classes minoritárias.
- *Two-Phase Fine-Tuning*: Warmup do classificador final com LR maior, seguido do descongelamento do último bloco convolucional da ResNet50V2 com LR minúsculo.
- Integração do novo módulo de métricas rigorosas (`src.evaluate.metrics`).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.utils import class_weight
import matplotlib.pyplot as plt

# Adiciona a raiz do projeto ao sys.path para importar o src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from src.evaluate.metrics import evaluate_model

In [ ]:
# Caminhos
train_dir = '../../data/raw/fer2013/train'
val_dir = '../../data/raw/fer2013/test'
models_dir = '../../models'
os.makedirs(models_dir, exist_ok=True)

In [ ]:
# Data Augmentation Rigoroso
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validação sem augmentation
val_datagen = ImageDataGenerator()

batch_size = 64

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(48, 48),
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(48, 48),
    color_mode='grayscale',
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
# Cálculo dos pesos das classes
classes = train_generator.classes
c_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)
c_weights = dict(enumerate(c_weights_array))
print("Pesos das Classes:", c_weights)

In [ ]:
# Construção da Arquitetura
base_model = ResNet50V2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # Congela a base inicialmente

inputs = Input(shape=(48, 48, 1))
x = tf.image.grayscale_to_rgb(inputs)
x = tf.image.resize(x, (224, 224))
x = tf.keras.applications.resnet_v2.preprocess_input(x)

x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(7, activation='softmax')(x)

model = Model(inputs, outputs)
model.summary()

In [ ]:
# FASE 1: Warmup do Top Classifier
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

callbacks_f1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
    ModelCheckpoint(os.path.join(models_dir, 'resnet50v2_fase1_best.h5'), save_best_only=True)
]

history1 = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    class_weight=c_weights,
    callbacks=callbacks_f1
)

In [ ]:
# FASE 2: Fine-Tuning do último bloco convolucional
base_model.trainable = True

# Congela tudo exceto as últimas 30 camadas (último bloco da ResNet50V2)
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

callbacks_f2 = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
    ModelCheckpoint(os.path.join(models_dir, 'resnet50v2_fase2_best.h5'), save_best_only=True)
]

history2 = model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    class_weight=c_weights,
    callbacks=callbacks_f2
)

In [ ]:
# Avaliação Final e Geração de Matriz de Confusão
x_val = []
y_val = []
val_generator.reset()
for i in range(len(val_generator)):
    x, y = next(val_generator)
    x_val.append(x)
    y_val.append(y)
x_val = np.concatenate(x_val)
y_val = np.concatenate(y_val)

classes = list(train_generator.class_indices.keys())
df_metrics = evaluate_model(model, x_val, y_val, classes, 'resnet50v2_advanced')
display(df_metrics)